# Step 2: Signal & Indicator Engineering

## 1. Objective & Alignment
Streamline raw 1-second market snapshots into the exact indicator suite required by the execution engine:
1. **Price Gap**: Difference between real-time Binance BTC price and Chainlink oracle "price to beat".
2. **RSI(14) on Gap**: 14-period Wilder Relative Strength Index measuring gap expansion/contraction momentum.
3. **ATR(14) on BTC**: 14-period Average True Range measuring underlying volatility.
4. **RTV(30) on BTC**: 30-period Real-Time Volatility measuring tick-level noise.
5. **Gap Safety**: Ratio of $|\text{gap}| / \text{ATR}$ (directional buffer relative to market volatility).
6. **Peak Gap Ratio**: Ratio of current gap to peak gap ($|\text{gap}| / \text{peakAbsGap}$).
7. **Order Book Depth & Imbalance**: Best bids, best asks, spreads, and order book imbalance (OBI).

All extra/unnecessary features have been removed to keep the pipeline streamlined and aligned with the engine.

## 2. Setup & Dependencies

In [1]:
import pandas as pd
import numpy as np
import json
import ast
from pathlib import Path
from datetime import datetime, timezone

try:
    from IPython.display import display
except ImportError:
    display = print

# Optional visualization libraries
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
    plt.style.use('seaborn-v0_8-whitegrid')
    plt.rcParams['figure.figsize'] = (14, 5)
except ImportError:
    HAS_PLOTS = False

RAW_DATA_DIR = Path('../data/market_data/raw')
PROCESSED_DATA_DIR = Path('../data/market_data/processed')
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

raw_files = sorted(RAW_DATA_DIR.rglob('*.csv'))
print(f'Found {len(raw_files)} raw data files:')
for f in raw_files:
    print(f'  - {f.name}')

Found 8 raw data files:
  - btc-updown-15m-1793942100.csv
  - btc-updown-5m-1793932800.csv
  - btc-updown-5m-1793933400.csv
  - btc-updown-5m-1793940600.csv
  - btc-updown-5m-1793941200.csv
  - btc-updown-5m-1793941500.csv
  - btc-updown-5m-1793941800.csv
  - btc-updown-5m-1794454500.csv


## 3. Load & Clean Raw Data

In [2]:
def load_and_clean_raw(data_dir: Path) -> pd.DataFrame:
    frames = []
    for p in sorted(data_dir.rglob('*.csv')):
        df_slot = pd.read_csv(p)
        frames.append(df_slot)
        
    df = pd.concat(frames, ignore_index=True)
    df['current_time'] = pd.to_datetime(df['current_time'])
    
    # Ensure numeric columns are floats
    num_cols = [
        'chainlink_price_to_beat', 'binance_current_price',
        'polymarket_up_best_bid', 'polymarket_up_best_ask', 'polymarket_up_spread',
        'polymarket_up_bid_liquidity', 'polymarket_up_ask_liquidity',
        'polymarket_down_best_bid', 'polymarket_down_best_ask', 'polymarket_down_spread',
        'polymarket_down_bid_liquidity', 'polymarket_down_ask_liquidity'
    ]
    for col in num_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
    # Forward fill any initial missing prices before stream sync
    df[['binance_current_price', 'chainlink_price_to_beat']] = df.groupby('market_slug')[['binance_current_price', 'chainlink_price_to_beat']].ffill()
    
    # Parse JSON depth
    def parse_depth(val):
        if isinstance(val, list):
            return val
        if pd.isna(val) or val == '':
            return []
        try:
            return json.loads(val)
        except Exception:
            return []
            
    for c in ['up_top5_bids', 'up_top5_asks', 'down_top5_bids', 'down_top5_asks']:
        df[c] = df[c].apply(parse_depth)
        
    return df

df = load_and_clean_raw(RAW_DATA_DIR)
print(f'Loaded dataset: {df.shape[0]} snapshots across {df["market_slug"].nunique()} slots.')

Loaded dataset: 726 snapshots across 8 slots.


## 4. Time Tracking Features

- `elapsed_seconds`: Seconds elapsed since window opened.
- `remaining_seconds`: Seconds remaining until market resolution.

In [3]:
# Extract Unix seconds
df['timestamp_unix'] = df['current_time'].apply(lambda x: x.timestamp())
df['elapsed_seconds'] = (df['timestamp_unix'] - df['slot_start']).round(2)
df['remaining_seconds'] = (df['slot_end'] - df['timestamp_unix']).round(2)

print('Time tracking sample:')
display(df[['market_slug', 'current_time', 'elapsed_seconds', 'remaining_seconds']].head(3))

Time tracking sample:


,market_slug,current_time,elapsed_seconds,remaining_seconds
0,btc-updown-15m-1793942100,2026-11-06 05:18:58.237321+00:00,238.24,61.76
1,btc-updown-15m-1793942100,2026-11-06 05:18:59.223518+00:00,239.22,60.78
2,btc-updown-15m-1793942100,2026-11-06 05:19:00.239039+00:00,240.24,59.76


## 5. Streamlined Indicator Suite

Implements the exact indicators required for trading decisions:
- **Price Gap**: $\text{Price}_{\text{Binance}} - \text{Price}_{\text{Oracle}}$
- **RSI(14) on Gap**: Wilder's smoothing measuring gap direction momentum.
- **ATR(14) on BTC**: Wilder's smoothing of 1-second price changes ($|\Delta P|$).
- **RTV(30) on BTC**: 30-period rolling average of absolute price changes.
- **Gap Safety**: $|\text{gap}| / \text{ATR}$
- **Peak Gap Ratio**: $|\text{gap}| / \text{peakAbsGap}$

In [4]:
# 1. Price Gap
df['price_gap'] = df['binance_current_price'] - df['chainlink_price_to_beat']
df['price_gap_abs'] = df['price_gap'].abs()

# 2. Wilder RSI(14) on Price Gap
def compute_wilder_rsi(series: pd.Series, period: int = 14) -> pd.Series:
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = (-delta).where(delta < 0, 0.0)
    avg_gain = gain.ewm(com=period - 1, min_periods=period).mean()
    avg_loss = loss.ewm(com=period - 1, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100.0 - (100.0 / (1.0 + rs))
    return rsi.fillna(50.0)

df['rsi_gap'] = df.groupby('market_slug')['price_gap'].transform(lambda x: compute_wilder_rsi(x, period=14))

# 3. Wilder ATR(14) on BTC Price
def compute_wilder_atr(series: pd.Series, period: int = 14) -> pd.Series:
    tr = series.diff().abs()
    return tr.ewm(com=period - 1, min_periods=period).mean()

df['atr_btc'] = df.groupby('market_slug')['binance_current_price'].transform(lambda x: compute_wilder_atr(x, period=14))

# 4. RTV(30) - Real-Time Volatility over 30-tick rolling window
def compute_rtv(series: pd.Series, window: int = 30) -> pd.Series:
    tr = series.diff().abs()
    return tr.rolling(window=window, min_periods=3).mean()

df['rtv_btc'] = df.groupby('market_slug')['binance_current_price'].transform(lambda x: compute_rtv(x, window=30))

# 5. Gap Safety: |gap| / ATR
df['gap_safety'] = np.where(
    (df['atr_btc'] > 0) & df['atr_btc'].notna(),
    df['price_gap_abs'] / df['atr_btc'],
    np.nan
)

# 6. Peak Gap Ratio: |gap| / peakAbsGap
df['peak_gap_abs'] = df.groupby('market_slug')['price_gap_abs'].cummax()
df['peak_gap_ratio'] = np.where(
    df['peak_gap_abs'] > 0,
    df['price_gap_abs'] / df['peak_gap_abs'],
    0.0
)

print('Computed Indicators Sample:')
cols = ['price_gap', 'rsi_gap', 'atr_btc', 'rtv_btc', 'gap_safety', 'peak_gap_ratio']
display(df[cols].describe().round(3))

Computed Indicators Sample:


,price_gap,rsi_gap,atr_btc,rtv_btc,gap_safety,peak_gap_ratio
count,726.000,726.000,614.000,702.000,614.000,726.000
mean,111.450,33.128,1.182,1.333,140.825,0.647
std,69.076,25.030,0.599,0.837,199.995,0.222
min,12.307,0.051,0.084,0.000,5.384,0.113
25%,52.652,7.783,0.769,0.814,56.856,0.518
50%,90.257,32.518,1.119,1.280,85.675,0.662
75%,159.749,50.000,1.532,1.612,150.106,0.828
max,304.739,93.770,3.532,6.760,1884.662,1.000


## 6. Order Book Depth & Microstructure Signals

- `up_midpoint`: Market-implied probability for UP outcome.
- `down_midpoint`: Market-implied probability for DOWN outcome.
- `up_obi` & `down_obi`: Order Book Imbalance $\frac{\text{BidLiq} - \text{AskLiq}}{\text{BidLiq} + \text{AskLiq}}$.

In [5]:
df['up_midpoint'] = (df['polymarket_up_best_bid'] + df['polymarket_up_best_ask']) / 2
df['down_midpoint'] = (df['polymarket_down_best_bid'] + df['polymarket_down_best_ask']) / 2

tot_up = df['polymarket_up_bid_liquidity'] + df['polymarket_up_ask_liquidity']
df['up_obi'] = np.where(tot_up > 0, (df['polymarket_up_bid_liquidity'] - df['polymarket_up_ask_liquidity']) / tot_up, 0.0)

tot_down = df['polymarket_down_bid_liquidity'] + df['polymarket_down_ask_liquidity']
df['down_obi'] = np.where(tot_down > 0, (df['polymarket_down_bid_liquidity'] - df['polymarket_down_ask_liquidity']) / tot_down, 0.0)

print('Order book signals summary:')
display(df[['up_midpoint', 'down_midpoint', 'up_obi', 'down_obi']].describe().round(3))

Order book signals summary:


,up_midpoint,down_midpoint,up_obi,down_obi
count,588.000,588.000,726.000,726.000
mean,0.510,0.490,-0.272,0.272
std,0.233,0.233,0.432,0.432
min,0.015,0.075,-1.000,-0.367
25%,0.385,0.315,-0.642,-0.104
50%,0.500,0.500,-0.175,0.175
75%,0.685,0.615,0.104,0.642
max,0.925,0.985,0.367,1.000


## 7. Save Streamlined Processed Datasets

Exports the clean, streamlined dataset with only necessary columns for the backtesting engine.

In [6]:
# Re-serialize JSON depth columns for CSV storage
df_out = df.copy()
for c in ['up_top5_bids', 'up_top5_asks', 'down_top5_bids', 'down_top5_asks']:
    df_out[c] = df_out[c].apply(lambda x: json.dumps(x) if isinstance(x, list) else x)
    
# Remove temporary columns
df_out = df_out.drop(columns=['timestamp_unix'], errors='ignore')

# Save per-slot files
for slug, group in df_out.groupby('market_slug'):
    out_file = PROCESSED_DATA_DIR / f'{slug}_features.csv'
    group.to_csv(out_file, index=False)
    print(f'Saved streamlined slot: {out_file.name} ({len(group)} rows, {len(group.columns)} cols)')
    
# Also save all_features.csv for combined analysis
all_file = PROCESSED_DATA_DIR / 'all_features.csv'
df_out.to_csv(all_file, index=False)
print(f'Saved combined features: {all_file.name} ({len(df_out)} rows, {len(df_out.columns)} cols)')

Saved streamlined slot: btc-updown-15m-1793942100_features.csv (300 rows, 35 cols)
Saved streamlined slot: btc-updown-5m-1793932800_features.csv (15 rows, 35 cols)
Saved streamlined slot: btc-updown-5m-1793933400_features.csv (16 rows, 35 cols)
Saved streamlined slot: btc-updown-5m-1793940600_features.csv (15 rows, 35 cols)
Saved streamlined slot: btc-updown-5m-1793941200_features.csv (15 rows, 35 cols)
Saved streamlined slot: btc-updown-5m-1793941500_features.csv (15 rows, 35 cols)
Saved streamlined slot: btc-updown-5m-1793941800_features.csv (300 rows, 35 cols)
Saved streamlined slot: btc-updown-5m-1794454500_features.csv (50 rows, 35 cols)
Saved combined features: all_features.csv (726 rows, 35 cols)


## 8. Summary & Indicator Inventory

| Indicator | Purpose | Usage in Trading Rules |
| :--- | :--- | :--- |
| `price_gap` | Binance BTC vs Oracle Open | Core directional lead signal |
| `rsi_gap` | Wilder RSI(14) on Price Gap | Momentum confirmation & stop-loss suppression |
| `atr_btc` | Wilder ATR(14) on BTC Price | Volatility regime filter (`atr <= 2.0`) |
| `rtv_btc` | Rolling 30-tick volatility | Tick noise measurement |
| `gap_safety` | $|\text{gap}| / \text{ATR}$ | High-confidence entry cushion (`gap_safety >= 40.0`) |
| `peak_gap_ratio` | $|\text{gap}| / \text{peakAbsGap}$ | Trend retention filter (`peak_gap_ratio >= 0.75`) |
| `up_obi` / `down_obi` | Order Book Imbalance | Microstructure liquidity pressure |

The dataset is now lean, aligned, and ready for backtesting in `03_backtesting.ipynb`.